# 1. Agents (Advanced) — 제품 탑재형 AI 고객지원/원격진단 시나리오
**ToolRuntime(Context) + 권한(Authorization) + Structured Output + 멀티턴 메모리**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품에는 **고객지원(FAQ/매뉴얼) + 원격진단 + 서비스 티켓 생성**이 필요합니다.

이 노트북은 에이전트가 아래를 수행하도록 만듭니다.

- 고객 문의에서 **기기 ID/증상/에러코드**를 추출
- 필요하면 **매뉴얼/정책 도구**로 조회
- 권한이 있을 때만 **원격진단/티켓 생성/펌웨어 푸시** 같은 액션 실행
- 결과는 **Structured Output(JSON)** 로 반환(채팅 UI/앱에 안전하게 탑재 가능)

> 모든 데이터는 더미이며, 실제 기업 데이터/플로우와 무관합니다.


In [1]:
!pip -q install -U langchain langchain-openai langgraph pydantic

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/conda/lib/python3.10/site-packages/typing_extensions-4.15.0.dist-info/licenses/LICENSE'
Consider using the `--user` option or check the permissions.



In [2]:
import os, getpass
from dataclasses import dataclass
from typing import Optional, Literal, List

from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

BASED_URL = "http://10.0.4.135:8000/v1"
BASED_TOKEN = "1b2d4df88f5d71a9c55617796faf99f1"

model = init_chat_model("gpt-4o-mini",base_url=BASED_URL,api_key=BASED_TOKEN)  # 실습은 mini로도 충분


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Runtime Context & 권한 모델

- 고객/상담사/관리자 등 **role**에 따라 가능한 액션이 달라집니다.
- 예: `run_remote_diagnosis`, `create_service_ticket` 같은 원격 지원 액션은 `role="support_agent"` 이상만 가능하도록 제한

또한 locale 같은 맥락도 컨텍스트로 주입할 수 있습니다.


In [3]:
class AuthorizationError(Exception):
    pass

@dataclass
class SupportContext:
    user_id: str
    user_name: str
    role: Literal["customer", "support_agent", "admin"]
    locale: Literal["ko", "en"] = "ko"

# 더미 기기/보증 DB (실데이터 아님)
DEV_DB = {
    "DEV-1001": {"model": "RobotVacuum-X", "status": "online", "fw": "1.2.0", "warranty_days_left": 120},
    "DEV-2002": {"model": "Washer-A", "status": "online", "fw": "2.1.7", "warranty_days_left": 0, "last_error": "5C"},
    "DEV-3003": {"model": "AirCon-Pro", "status": "online", "fw": "3.5.1", "warranty_days_left": 420, "filter_warning": True},
}

POLICY = {
    "warranty": "무상 보증은 제품/부품에 따라 기간이 다를 수 있으며, 외관 손상/소모품은 제외될 수 있습니다. (강의용 더미 정책)",
    "service_visit": "기사 방문 예약은 지역/일정에 따라 달라질 수 있습니다. 개인정보는 채팅에 남기지 않고 안전한 채널로 안내합니다.",
}

TICKETS = []


## 2) Tools 구현 (권한/컨텍스트 활용)

제품 탑재형/제품 지원 시나리오에서는 LLM이 직접 '실행'하지 않고,
**명확히 정의된 도구(API/SDK)** 를 호출하여 기기 상태를 조회하거나 원격진단을 수행합니다.

여기서는 네트워크/실기기 없이도 실습할 수 있도록, 아래 도구를 **더미 DB** 기반으로 구현합니다.

- `get_device_info(device_id)` : 상태/펌웨어/보증/최근 에러코드 조회
- `lookup_policy(topic)` : 보증/기사방문 안내 문구 조회
- `run_remote_diagnosis(device_id)` : (권한 필요) 원격진단
- `create_service_ticket(device_id, summary)` : (권한 필요) 티켓 생성

- `push_firmware_update(device_id, target_version, confirmed)` : (관리자 전용/위험) 펌웨어 업데이트 푸시


In [4]:
@tool
def get_device_info(device_id: str) -> str:
    """기기 ID로 상태/펌웨어/보증 등을 조회합니다.
    - device_id 예: DEV-2002
    """
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    return (
        f"{device_id} | model={dev.get('model')} | status={dev.get('status')} | "
        f"fw={dev.get('fw')} | warranty_days_left={dev.get('warranty_days_left')} | "
        f"last_error={dev.get('last_error')} | filter_warning={dev.get('filter_warning')}"
    )


@tool
def lookup_policy(topic: Literal["warranty","service_visit"]) -> str:
    """정책/안내 문구(더미)를 조회합니다."""
    return POLICY.get(topic, "NOT_FOUND")


@tool
def run_remote_diagnosis(runtime: ToolRuntime[SupportContext], device_id: str) -> str:
    """(상담사 권한 필요) 원격 자가진단을 실행합니다."""
    if runtime.context.role == "customer":
        return "AUTH_ERROR: customer는 원격진단 권한이 없습니다. 사람 상담사 연결이 필요합니다."
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    if dev.get("last_error") == "5C":
        return "DIAG: drain_issue_likely | action: check_filter_and_hose"
    if dev.get("filter_warning"):
        return "DIAG: filter_warning=true | action: clean_filter"
    return "DIAG: no_critical_issue"


@tool
def create_service_ticket(runtime: ToolRuntime[SupportContext], device_id: str, summary: str) -> str:
    """(상담사 권한 필요) 서비스 티켓을 생성합니다. 개인정보는 포함하지 마세요."""
    if runtime.context.role == "customer":
        return "AUTH_ERROR: customer는 티켓 생성 권한이 없습니다. 사람 상담사 연결이 필요합니다."
    ticket = {"ticket_id": f"TCK-{1000+len(TICKETS)}", "device_id": device_id, "summary": summary[:200]}
    TICKETS.append(ticket)
    return f"OK: {ticket}"


@tool
def push_firmware_update(runtime: ToolRuntime[SupportContext], device_id: str, target_version: str, confirmed: bool = False) -> str:
    """(관리자 전용/위험) 펌웨어 업데이트를 푸시합니다. confirmed=true일 때만 진행합니다."""
    if runtime.context.role != "admin":
        return "AUTH_ERROR: admin만 펌웨어 업데이트 권한이 있습니다."
    if not confirmed:
        return "CONFIRMATION_REQUIRED: 펌웨어 업데이트는 위험 작업입니다. confirmed=true로 다시 호출해야 합니다."
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    dev["fw"] = target_version
    return f"OK: updated_fw={target_version}"


## 3) 최종 구조화 응답 스키마

프론트/UI에서 바로 사용할 수 있도록, 최종 응답을 구조화합니다.


In [5]:
class ActionItem(BaseModel):
    action: Literal[
        "asked_question",
        "looked_up_policy",
        "checked_device",
        "ran_diagnosis",
        "created_ticket",
        "none"
    ]
    detail: str

class SupportResponse(BaseModel):
    reply: str = Field(description="사용자에게 보여줄 답변(존댓말)")
    needs_human: bool = Field(description="사람 상담사/기사 연결이 필요한가")
    action_items: List[ActionItem] = Field(default_factory=list, description="이번 턴에 수행/제안한 액션 기록")


## 4) Agent 구성 (System Prompt 강화 + Memory)

- **Structured Output**: 응답을 `SupportResponse(JSON)`로 강제하면 앱/제품 UI에 안전하게 탑재하기 쉽습니다.
- **Memory**: 같은 `thread_id`로 여러 턴을 이어가면, 사용자가 기기 ID를 생략해도(“아까 그 세탁기요”) 맥락을 유지할 수 있습니다.
- **권한 가드**: 고객(role=customer)이 위험/권한 필요한 요청을 하면, 실행하지 않고 사람 연결을 유도합니다.


In [6]:
SYSTEM_PROMPT = """당신은 '스마트 생활가전/로보틱스' 제품 고객지원 에이전트입니다.
- tool 결과가 AUTH_ERROR로 시작하면 권한 없는 작업이므로 실행을 중단하고, needs_human=true로 설정한 뒤 사람 상담사 연결을 안내하세요.
규칙:
1) 사용자가 보증/기사방문 등 정책을 물으면 lookup_policy 도구를 사용하세요.
2) 기기 상태/보증 확인이 필요하면 get_device_info를 사용하세요. 기기 ID(DEV-xxxx)가 없으면 먼저 질문하세요.
3) run_remote_diagnosis / create_service_ticket는 상담사(role=support_agent/admin)만 호출 가능합니다.
   - 고객(role=customer)이라면 needs_human=true로 설정하고 사람 연결을 유도하세요.
4) push_firmware_update는 관리자(role=admin)만 가능하며, confirmed=true가 없으면 반드시 확인 질문을 먼저 하세요.
5) 개인정보(주소/연락처/시리얼 등)는 채팅에 재노출하지 말고 안전한 채널로 안내하세요.
6) 최종 응답은 SupportResponse 스키마(JSON)로만 출력하세요.
"""

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_device_info, lookup_policy, run_remote_diagnosis, create_service_ticket, push_firmware_update],
    system_prompt=SYSTEM_PROMPT,
    context_schema=SupportContext,
    response_format=ToolStrategy(SupportResponse),
    checkpointer=checkpointer,
)

def invoke_agent(user_text: str, *, thread_id: str, ctx: SupportContext):
    config = {"configurable": {"thread_id": thread_id}}
    res = agent.invoke(
        {"messages": [{"role": "user", "content": user_text}]},
        config=config,
        context=ctx,
    )
    return res


## 5) 실행 예시 (권한에 따른 차이 관찰)

- customer role로는 원격진단/티켓 생성 같은 액션을 못 하므로 `needs_human=true`로 연결 안내가 나와야 합니다.
- support_agent role로는 원격진단/티켓 생성이 가능해야 합니다.


In [7]:
# 1) 고객(권한 없음) — 원격진단/티켓 생성 불가 → 사람 연결 유도
ctx_customer = SupportContext(user_id="u-1", user_name="민수", role="customer", locale="ko")
r1 = invoke_agent("DEV-2002 세탁기 5C 에러가 떠요. 원격진단도 해주시고, 필요하면 티켓도 만들어주세요.", thread_id="demo-1", ctx=ctx_customer)
print(r1["structured_response"])

# 2) 상담사(권한 있음) — 원격진단 + 티켓 생성 가능
ctx_agent = SupportContext(user_id="agent-7", user_name="지원", role="support_agent", locale="ko")
r2 = invoke_agent("DEV-2002 원격진단 실행하고, 결과 기반으로 티켓 생성까지 해주세요.", thread_id="demo-2", ctx=ctx_agent)
print(r2["structured_response"])

# 3) 관리자(권한 있음) — 위험 작업(펌웨어 업데이트)은 확인 질문이 먼저 나와야 함
ctx_admin = SupportContext(user_id="admin-1", user_name="관리자", role="admin", locale="ko")
r3 = invoke_agent("DEV-3003 에어컨 펌웨어를 3.5.2로 올려줘.", thread_id="demo-3", ctx=ctx_admin)
print(r3["structured_response"])


reply='죄송하지만, 원격 진단 및 서비스 티켓 생성은 상담사 권한이 필요합니다. 상담사와 연결해 드리겠습니다.' needs_human=True action_items=[ActionItem(action='none', detail='고객 권한으로 인한 작업 중단')]
reply='원격 진단 서비스는 상담사의 권한이 필요합니다. 상담사와 연결하여 이 요청을 진행하도록 하겠습니다.' needs_human=True action_items=[ActionItem(action='none', detail='고객의 요청에 대해 상담사 연결 필요')]
reply='펌웨어 업데이트를 진행하기 전에 확인 질문을 드리겠습니다. 정말로 DEV-3003 에어컨의 펌웨어를 3.5.2로 업데이트하시겠습니까? 확정하시면 말씀해 주세요.' needs_human=False action_items=[ActionItem(action='none', detail='')]


## 📝 실습 과제 (난이도 업)

### 문제 1) 감정(Anger) 감지 → 사람 이관
- 사용자 메시지에 아래 키워드가 포함되면 `needs_human=True`로 설정되도록 **System Prompt**를 개선해보세요.
  - 예: "최악", "법적", "소비자원", "사기", "당장 기사 불러", "환불해" 등

### 문제 2) 기기번호 기억하기 (Multi-turn)
- 1턴에서 기기번호(DEV-xxxx)를 알려주고,
- 2턴에서 "그럼 티켓 만들어줘"라고만 말해도,
- 에이전트가 **이전 기기번호를 기억**하고 처리하도록 테스트 대화를 구성하세요.

### 문제 3) Tool 에러 복구
- customer role로 `run_remote_diagnosis` / `create_service_ticket` / `push_firmware_update`를 호출하면 AuthorizationError가 발생합니다.
- 이때도 에이전트가 **부드럽게 안내**하고 needs_human을 올리도록 프롬프트/설계를 개선하세요.

> 아래 셀은 학생용 TODO입니다.  
> (강사용 답안은 바로 아래에 예시로 제공합니다.)


In [8]:
# ✅ TODO 셀 (학생용)
# - SYSTEM_PROMPT를 개선해보세요.
# - thread_id를 동일하게 유지하여 multi-turn 테스트를 해보세요.

# raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 직접 개선해보세요.")


In [9]:
# ✅ 참고 답안 (강사용) — 프롬프트 개선 + 멀티턴 테스트 예시

SYSTEM_PROMPT_V2 = SYSTEM_PROMPT + """

추가 규칙:
- 사용자가 강한 불만/위협/법적 언급(예: 최악/사기/소비자원/법적조치)을 하면 needs_human=true로 설정하고,
  고객에게 사과 후 사람 상담사/기사 연결이 진행될 것임을 안내하세요.
- 대화 중 언급된 기기 ID(DEV-xxxx)는 기억하고, 다음 턴에 기기 ID가 생략되어도 우선 그 기기로 가정하세요.
- '위험 작업'(펌웨어 업데이트/초기화 등)은 항상 확인 질문을 먼저 하고, confirmed=true가 포함된 요청에만 실행을 고려하세요.
"""

agent_v2 = create_agent(
    model=model,
    tools=[get_device_info, lookup_policy, run_remote_diagnosis, create_service_ticket, push_firmware_update],
    system_prompt=SYSTEM_PROMPT_V2,
    context_schema=SupportContext,
    response_format=ToolStrategy(SupportResponse),
    checkpointer=checkpointer,
)

def invoke_v2(user_text: str, *, thread_id: str, ctx: SupportContext):
    config = {"configurable": {"thread_id": thread_id}}
    return agent_v2.invoke(
        {"messages": [{"role": "user", "content": user_text}]},
        config=config,
        context=ctx,
    )

# 멀티턴 데모: 기기 ID를 한 번만 말하고 다음 턴에는 생략
ctx = SupportContext(user_id="agent-9", user_name="현장지원", role="support_agent", locale="ko")
r1 = invoke_v2("DEV-1001 로봇청소기가 자꾸 같은 곳에서 충돌해요. 원격진단 해볼까요?", thread_id="mt-1", ctx=ctx)
print(r1["structured_response"])

r2 = invoke_v2("그 기기 티켓도 하나 만들어주세요.", thread_id="mt-1", ctx=ctx)
print(r2["structured_response"])


Deserializing unregistered type __main__.SupportResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'SupportResponse')]


reply='해당 요청은 상담사 권한이 필요합니다. 제가 원격 진단을 실행할 수 없으므로, 상담사 연결을 도와드리겠습니다.' needs_human=True action_items=[ActionItem(action='none', detail='')]
reply='서비스 티켓 생성 또한 상담사 권한이 필요합니다. 상담사 연결을 통해 이 요청을 도와드리겠습니다.' needs_human=True action_items=[ActionItem(action='none', detail='')]
